# BSV Batch Processor — Automatic Construction (text/table only)

Runs the full corpus (`/projects/agrifutur/bsv_pdfs` + `/projects/agrifutur/bsv_pdfs_all`),
**excluding the PDFs reserved for GT**, producing `text_table` pairs
only. Images disabled entirely for this run.

Uses `bsv_parser.docling_extract_tables`, which now has the strict
quality heuristics (row/column minimums, empty-cell %, H38, H39)
built directly into it, ported from the strict pdfplumber-based
`extract_tables()` and confirmed via direct testing on a 30-PDF
sample: 21/75 low-quality tables correctly dropped, no false positives
observed.

**Honest cost note**: Docling processes every page of every PDF, real
measured throughput tonight ranged from ~2 PDFs/min (4 CPUs) to ~13
PDFs/min (RTX 8000, steady-state). Across the full ~20,000-PDF corpus
this is a multi-hour to multi-day job depending on hardware. Checkpoint
makes it safe to interrupt and resume.


In [1]:
import json, os, sys, threading
import fitz, pdfplumber
from concurrent.futures import ThreadPoolExecutor, as_completed

## Compile `bsv_parser.py` from the real source notebook

Same compile step used by every real batch tonight, reads
`bsv_parser_subsections_v3.ipynb` from the working directory.


In [2]:
nb = json.load(open("/projects/agrifutur/yabusaqe/BSV Sample Construction/Parsers/bsv_parser_subsections_v3.ipynb"))
code_cells = [c for c in nb["cells"] if c["cell_type"] == "code"]

with open("/projects/agrifutur/yabusaqe/BSV Sample Construction/Parsers/bsv_parser.py", "w") as f:
    for cell in code_cells:
        src = "".join(cell["source"])
        if any(skip in src for skip in [
            "pip install",
            "parse_pdf(TEST_PDF",
            "urllib.request.urlretrieve",
            "fitz.open(TEST_PDF",
            "fitz.open(TEST_CHARENTES",
            "parse_structure(TEST_PDF",
            "os.listdir(PDF_DIR)",
            "parse_pdf(TEST_PDF2",
            "parse_pdf(TEST_CHARENTES",
            "print(SIZE_THRESHOLD)",
        ]):
            continue
        f.write(src + "\n\n")

print("Created bsv_parser.py")
print(os.path.exists("bsv_parser.py"))

Created bsv_parser.py
True


In [3]:
if "bsv_parser" in sys.modules:
    import importlib
    import bsv_parser
    importlib.reload(bsv_parser)
sys.path.insert(0, os.getcwd())
import bsv_parser
from bsv_parser import parse_pdf
print("bsv_parser loaded")

/projects/agrifutur/yabusaqe/BSV Sample Construction
['test_bulletin.pdf', '.ipynb_checkpoints', 'bsv_parser.ipynb', '__pycache__', 'PDF_URLS.txt', 'Previous Iterations Data', 'bsv_heuristics_analysis.ipynb', 'bsv_download.ipynb', 'bsv_parser_subsections.ipynb', 'bsv_parser_subsections_v2.ipynb', 'french_words.txt', 'recall_approx.py', 'reserve_gt_pdfs.py', 'gt_reserved_pdfs.jsonl', 'Old Viewers', 'bsv_batch.ipynb', 'bsv_pair_constructor.py', 'bsv_parser.py', 'bsv_parser_subsections_v3.ipynb', 'generate_reviewer.py', 'bsv_images_gt_2404', 'bsv_batch_tables_docling.ipynb', 'bsv_images_gt_tables_docling', 'bsv_dataset_gt_tables_docling.jsonl', 'bsv_pairs_gt_texttable_900.jsonl', 'generate_reviewer_V2.py', 'bsv_dataset_gt_tables_realspans.jsonl', 'processed_urls_gt_tables_docling.txt', 'bsv_pairs_gt_texttable_partial.jsonl', 'bsv_reviewer_texttable_partial.html', 'bsv_validations_export.json', 'bsv_dataset_gt_2404.jsonl', 'bsv_pairs_gt_textimage_900.jsonl', 'processed_urls_gt_2404.txt', '

## Override 1: tables via Docling (strict heuristics already built in)

No extra filtering added here, `docling_extract_tables` itself now
carries the row/column minimums, empty% check, H38, and H39.


In [4]:
bsv_parser.extract_tables = bsv_parser.docling_extract_tables

## Override 2: skip images entirely, this run is table-only

In [5]:
def skip_images(pdf_path, out_img_dir, bulletin_id, images_raw, text_els=None):
    return []

bsv_parser.extract_images = skip_images

os.environ["SKIP_OCR"] = "1"

## Configuration

`GT_MANIFEST` is the exclusion list, this run processes every PDF in
the two real source folders **except** those reserved for GT.


In [6]:
PDF_DIRS     = [
    "/projects/agrifutur/bsv_pdfs",
    "/projects/agrifutur/bsv_pdfs_all",
]
GT_MANIFEST  = "/projects/agrifutur/yabusaqe/BSV Sample Construction/Ground Truth/gt_reserved_pdfs.jsonl
OUT_JSONL    = "/projects/agrifutur/yabusaqe/BSV Sample Construction/Automatic Construction/bsv_dataset_auto_texttable.jsonl""
WORKERS      = 8
LIMIT        = None
CHECKPOINT   = "/projects/agrifutur/yabusaqe/BSV Sample Construction/Automatic Construction/Notebooks/processed_urls_auto_texttable.txt""
MIN_PDF_SIZE = 5000

## Build the file list, excluding every GT-reserved PDF

In [7]:
with open(GT_MANIFEST, encoding="utf-8") as f:
    gt_reserved = set(json.loads(line)["filename"] for line in f)

print(f"{len(gt_reserved)} PDFs reserved for GT, will be excluded from this run")

pdf_paths = {}
for pdf_dir in PDF_DIRS:
    for fname in os.listdir(pdf_dir):
        if not fname.endswith(".pdf"):
            continue
        if fname in gt_reserved:
            continue
        full_path = os.path.join(pdf_dir, fname)
        if os.path.getsize(full_path) < MIN_PDF_SIZE:
            continue
        pdf_paths[fname] = full_path

pdf_files = sorted(pdf_paths.keys())
if LIMIT:
    pdf_files = pdf_files[:LIMIT]

print(f"{len(pdf_files)} PDFs to process for automatic construction")

2404 PDFs reserved for GT, will be excluded from this run
17758 PDFs to process for automatic construction


## Process, same checkpoint and threading-lock pattern as the real batches

In [8]:
def process_one(filename):
    pdf_path = pdf_paths[filename]
    if os.path.getsize(pdf_path) < MIN_PDF_SIZE:
        return filename, []
    try:
        records = parse_pdf(pdf_path, "/tmp/unused_img_dir", url=None)
        return filename, records
    except Exception as e:
        return filename, []

In [ ]:
write_lock      = threading.Lock()
checkpoint_lock = threading.Lock()
counts = {"processed": 0, "sections": 0, "errors": 0}

if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f:
        done_files = set(f.read().splitlines())
    print(f"Resuming: {len(done_files)} PDFs already processed")
else:
    done_files = set()

files_to_run = [f for f in pdf_files if f not in done_files]
print(f"Remaining: {len(files_to_run)} PDFs to process")

with open(OUT_JSONL, "a", encoding="utf-8") as out_f:
    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        futures = {executor.submit(process_one, fname): fname for fname in files_to_run}
        for future in as_completed(futures):
            fname, records = future.result()
            with write_lock:
                for rec in records:
                    out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                out_f.flush()
                counts["sections"] += len(records)
                counts["processed"] += 1
                if not records:
                    counts["errors"] += 1
            with checkpoint_lock:
                with open(CHECKPOINT, "a") as ck:
                    ck.write(fname + "\n")
            if counts["processed"] % 50 == 0 or counts["processed"] <= 5:
                print(f"[{counts['processed']}/{len(files_to_run)}] "
                      f"sections={counts['sections']} errors={counts['errors']}")

print(f"Done. Processed={counts['processed']} Sections={counts['sections']} Errors={counts['errors']}")

## Build text_table pairs only

In [ ]:
from bsv_pair_constructor import generate_pairs

OUT_PAIRS = "/projects/agrifutur/yabusaqe/BSV Sample Construction/Automatic Construction/bsv_pairs_auto_texttable.jsonl"

all_pairs = []
with open(OUT_JSONL, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        pairs = generate_pairs(rec)
        all_pairs.extend(p for p in pairs if p["pair_type"] == "text_table")

print(f"{len(all_pairs)} text_table pairs built")

with open(OUT_PAIRS, "w", encoding="utf-8") as f:
    for p in all_pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

print(f"Written to {OUT_PAIRS}")